# Conditional StyleGAN2-ADA fine-tunes on APTOS-256 (Kaggle T4 x2)

Each version is one session (~10 h) of the queue **f100-ada → f010-ada → f010-noaug (ablation) → f025-ada → f050-ada**,
every GAN trained to the same total kimg. The logic lives in the repo (`src/gan/kaggle_train.py`, settings in
`notebooks/kaggle_session.json`), so a git push changes the next session while this notebook stays the same.

Before every *Save Version → Save & Run All (Commit)*:
1. **Input: the genai-data-prep output** (version 4 or later) whose `splits/aptos_splits.csv` equals the repo's committed
   split (checked: the f010-f050 zips depend on it, f100 must hold exactly the committed train ids).
2. **Input: the newest output of this notebook** (snapshots, `final/`, `ledger.json`). Inputs are pinned to a version:
   in the Input panel use **⋮ → Check for updates → Update** on `genai-gan-train` after every finished session.
   Without any previous output the session refuses to start (`first_session` in the JSON is only for the very first one).
3. Settings: Accelerator **GPU T4 x2**, **Internet ON**. The runner prints `RUNNER_EXIT 0` when all went well.

The newest version's output always holds every finished GAN (`final/<tag>_kimg<N>.pkl`) plus the running one's
last two snapshots; the eval notebook attaches only that output.

In [ ]:
REPO = "https://github.com/nabirakhan/genai"
OVERRIDES = []        # e.g. ["--hours", "6"] or ["--run", "f010-noaug"]; empty = notebooks/kaggle_session.json

In [ ]:
import os, sys, glob, re, json, time, shutil, signal, subprocess
WORK = "/kaggle/working"; CODE = f"{WORK}/genai"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--recurse-submodules"], check=False)
os.chdir(CODE)
subprocess.run([sys.executable, "scripts/apply_patches.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "click", "requests", "tqdm", "ninja",
                "psutil", "scipy"], check=True)
for opt in ("pyspng", "imageio-ffmpeg"):    # speed-ups only; stylegan3 falls back without them
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", opt]).returncode:
        print("optional package not installed:", opt)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch; print("torch", torch.__version__, "gpus", torch.cuda.device_count())
def sh(cmd, **kw):
    print("$", " ".join(cmd)); return subprocess.run(cmd, check=True, **kw)


In [ ]:
r = subprocess.run([sys.executable, "-m", "src.gan.kaggle_train", *OVERRIDES])
print("runner exit code", r.returncode)

In [ ]:
from PIL import Image as PImage
from IPython.display import display
for d in sorted(glob.glob(f"{WORK}/runs/*")):
    fakes = sorted(glob.glob(f"{d}/fakes[0-9]*.png"))       # fakes_init.png would sort last
    print(os.path.basename(d), "->", os.path.basename(fakes[-1]) if fakes else "no fakes yet")
    if fakes:
        im = PImage.open(fakes[-1]); im.thumbnail((1536, 1536)); display(im)   # full grid is 7680x4096
print(open(f"{WORK}/ledger.json").read() if os.path.exists(f"{WORK}/ledger.json") else "no ledger.json")
for root, _, files in os.walk(WORK):
    for f in files:
        p = os.path.join(root, f)
        if os.path.getsize(p) > 5e7: print(f"{os.path.getsize(p) / 1e6:8.0f} MB  {p}")
os.chdir(WORK)
shutil.rmtree(f"{WORK}/data", ignore_errors=True)
shutil.rmtree(CODE, ignore_errors=True)
if r.returncode == 4:
    raise RuntimeError("the runner trained nothing (refused or misconfigured inputs): see its log above")